In [19]:
from mc_experiment import (
    make_seed_counter,
    next_seed,
    standardize_innovations,
    summarize_reference_experiment,
    summarize_mle_augmentation_experiment,
    augmented_config_path,
)

from SymbolicDSGE import ModelParser, DSGESolver, Shock
from SymbolicDSGE.bayesian import make_prior

from numpy import log
import numpy as np

from scipy.stats import chi2, gaussian_kde, norm

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import cProfile

import contextlib
import io
STDOUT_VOID = lambda: contextlib.redirect_stdout(io.StringIO())

_KNOWN_R = True
_AUGMENTED_PARAM = 'Pi_coef'
_AUGMENTED_EQUATION = 'OutGap'
_AUGMENTED_CONFIG = augmented_config_path(_AUGMENTED_EQUATION)
_MEAS_ERR_SCALE = 0.01
_MC_SAMPLES = 100_000
_MC_ALPHA = 0.05
_MC_SUMMARY_ONLY = False
_MC_INCLUDE_BY_PREDICTOR = False
_FIGSIZE_1D = (10, 6)
_FIGSIZE_2D = (12, 6)



In [20]:
# Load reference model
parser = ModelParser("../../MODELS/misspec_test/reference.yaml")
config, kalman = parser.get_all()
solver = DSGESolver(config, kalman)

comp = solver.compile(
    n_state=3,
    n_exog=3,
)
sol = solver.solve(
    comp,
    steady_state=[0.0, 0.0, 0.0, 0.0, 0.0],
)

print("Transition matrix:\n", sol.A.round(3), "\n")
print("Shock Loadings:\n", sol.B.round(3))

Transition matrix:
 [[ 0.83  -0.     0.     0.     0.   ]
 [ 0.     0.85   0.     0.     0.   ]
 [ 0.288 -0.047  0.28   0.     0.   ]
 [ 0.892  0.708 -1.711  0.     0.   ]
 [ 0.7   -0.115 -1.363  0.     0.   ]] 

Shock Loadings:
 [[ 1.     0.     0.   ]
 [ 0.     1.     0.   ]
 [ 0.     0.     1.   ]
 [ 3.193  0.493 -6.107]
 [ 2.531 -0.406 -4.864]]


In [21]:
# Load Misspecified DGP
parser_dgp = ModelParser("../../MODELS/misspec_test/misspec.yaml")
config_dgp, kalman_dgp = parser_dgp.get_all()
solver_dgp = DSGESolver(config_dgp, kalman_dgp)
comp_dgp = solver_dgp.compile(
    n_state=3,
    n_exog=3,
)
sol_dgp = solver_dgp.solve(
    comp_dgp,
    steady_state=[0.0, 0.0, 0.0, 0.0, 0.0],
)

In [22]:
# Large sample simulations used to approximate measurement-noise variances
_large_sample_seed_counter = make_seed_counter(start=100_000)
shocks_large = {
    "g,z": Shock(10_000, "norm", multivar=True, seed=next_seed(_large_sample_seed_counter)).shock_generator(),
    "r": Shock(10_000, "norm", multivar=False, seed=next_seed(_large_sample_seed_counter)).shock_generator(),
}

sim1 = sol_dgp.sim(
    T=10_000,
    shocks=shocks_large,
    observables=True,
)

sim2 = sol.sim(
    T=10_000,
    shocks=shocks_large,
    observables=True,
)

In [23]:
T = 200
_plot_seed_counter = make_seed_counter(start=2_000_000)

err_var = np.var(np.column_stack([sim1["OutGap"], sim1["Infl"], sim1["Rate"]]), axis=0)
mc_reference = summarize_reference_experiment(
    sol,
    sol_dgp,
    T=T,
    err_var=err_var,
    meas_err_scale=_MEAS_ERR_SCALE,
    mc_samples=_MC_SAMPLES,
    known_r=_KNOWN_R,
    alpha=_MC_ALPHA,
    summary_only=_MC_SUMMARY_ONLY,
    include_by_predictor=_MC_INCLUDE_BY_PREDICTOR,
)

rep_ref = mc_reference["representative"]
sim_dgp = rep_ref.sim_dgp
obs = rep_ref.obs
kf = rep_ref.kf
std_innov = rep_ref.std_innov
err_scale = rep_ref.err_scale
N, n_obs = kf.innov.shape

_measurement_order = {"OutGap": 0, "Infl": 1, "Rate": 2}
_predictor_order = {"Pi": 0, "x": 1, "r": 2}

def _sort_summary(df):
    out = df.copy()
    if "measurement" in out.columns:
        out["measurement_order"] = out["measurement"].map(_measurement_order)
    if "predictor" in out.columns:
        out["predictor_order"] = out["predictor"].map(_predictor_order)
    if "target" in out.columns:
        out["target_order"] = out["target"].map(_predictor_order)
    if "regressor" in out.columns:
        out["regressor_order"] = out["regressor"].map(_predictor_order)
    sort_cols = [
        col
        for col in ["measurement_order", "target_order", "predictor_order", "regressor_order"]
        if col in out.columns
    ]
    if sort_cols:
        out = out.sort_values(sort_cols)
    return out.drop(columns=[col for col in ["measurement_order", "target_order", "predictor_order", "regressor_order"] if col in out.columns])

sim_ref = sol.sim(
    T=T,
    shocks={
        "g,z": Shock(T, "norm", multivar=True, seed=next_seed(_plot_seed_counter)).shock_generator(),
        "r": Shock(T, "norm", multivar=False, seed=next_seed(_plot_seed_counter)).shock_generator(),
    },
    observables=True,
)
ref = np.column_stack([sim_ref["OutGap"], sim_ref["Infl"], sim_ref["Rate"]])[1:, :]

obs_dgp = np.column_stack([sim1["OutGap"], sim1["Infl"], sim1["Rate"]])[1:, :]
if np.any(err_scale != 0.0):
    _plot_rng = np.random.default_rng(next_seed(_plot_seed_counter))
    obs_dgp = obs_dgp + _plot_rng.normal(scale=np.sqrt(err_scale), size=obs_dgp.shape)

In [24]:
print(f"Known R assumption: {_KNOWN_R}")
print(f"Augmented measurement equation: {_AUGMENTED_EQUATION}")
print(f"Augmented coefficient: {_AUGMENTED_PARAM}")
print(f"Monte Carlo replications: {_MC_SAMPLES}")
print("Noise Covariance:\n", np.diag(err_scale).round(3))
print("Error Variance:\n", err_var.round(3))

Known R assumption: True
Augmented measurement equation: OutGap
Augmented coefficient: Pi_coef
Monte Carlo replications: 100000
Noise Covariance:
 [[0.121 0.    0.   ]
 [0.    0.168 0.   ]
 [0.    0.    0.008]]
Error Variance:
 [12.082 16.786  0.779]


In [25]:
print(f"LB Test summary across {_MC_SAMPLES} replications:")
display(mc_reference["lb_summary"].round(3))

LB Test summary across 100000 replications:


,measurement,lb_stat,p_value,mc_se_lb_stat,mc_se_p_value,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high
0,OutGap,6.935,0.063,0.014,0.000,100000,72369,0.724,0.001,0.721,0.726
1,Infl,7.868,0.044,0.015,0.000,100000,79250,0.792,0.001,0.790,0.795
2,Rate,1.573,0.414,0.006,0.001,100000,11885,0.119,0.001,0.117,0.121


In [26]:
print(f"Moment Tests summary across {_MC_SAMPLES} replications:")
display(mc_reference["moment_specification_test_summary"].round(3))

Moment Tests summary across 100000 replications:


,test,distance,stat,p_value,mc_se_distance,mc_se_stat,mc_se_p_value,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high,df,sample_size,bandwidth
0,mean_zero_hac,0.120,2.860,0.535,0.000,0.009,0.001,100000,5673,0.057,0.001,0.055,0.058,3.0,200,4
1,cov_identity,8.567,1022.581,0.000,0.003,1.334,0.000,100000,100000,1.000,0.000,1.000,1.000,6.0,200,4


In [27]:
print("Innovations on orthogonalized predicted states (Monte Carlo averages and rejection rates):")
_sort_summary(mc_reference["measurement_regressions_orthogonalized_summary"]).round(3)

Innovations on orthogonalized predicted states (Monte Carlo averages and rejection rates):


,measurement,predictor,coef,standardized_coef,std_error,t_stat,p_value,r2,mc_se_coef,mc_se_standardized_coef,mc_se_std_error,mc_se_t_stat,mc_se_p_value,mc_se_r2,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high
0,OutGap,Pi,-2.234,-0.096,1.664,-1.365,0.278,0.014,0.005,0.0,0.001,0.003,0.001,0.0,100000,26985,0.270,0.001,0.267,0.273
1,OutGap,x,-0.168,-0.046,0.246,-0.648,0.450,0.007,0.001,0.0,0.000,0.003,0.001,0.0,100000,8464,0.085,0.001,0.083,0.086
2,OutGap,r,-1.039,-0.035,2.039,-0.489,0.477,0.006,0.006,0.0,0.001,0.003,0.001,0.0,100000,6552,0.066,0.001,0.064,0.067
3,Infl,Pi,-0.322,-0.010,1.966,-0.149,0.492,0.005,0.006,0.0,0.001,0.003,0.001,0.0,100000,5548,0.055,0.001,0.054,0.057
4,Infl,x,-0.001,0.001,0.290,0.009,0.494,0.005,0.001,0.0,0.000,0.003,0.001,0.0,100000,5432,0.054,0.001,0.053,0.056
5,Infl,r,-0.371,-0.009,2.400,-0.129,0.493,0.005,0.008,0.0,0.001,0.003,0.001,0.0,100000,5514,0.055,0.001,0.054,0.057
6,Rate,Pi,-0.020,-0.004,0.365,-0.063,0.497,0.005,0.001,0.0,0.000,0.003,0.001,0.0,100000,5204,0.052,0.001,0.051,0.053
7,Rate,x,0.012,0.015,0.054,0.214,0.489,0.005,0.000,0.0,0.000,0.003,0.001,0.0,100000,5733,0.057,0.001,0.056,0.059
8,Rate,r,-0.016,-0.001,0.445,-0.016,0.497,0.005,0.001,0.0,0.000,0.003,0.001,0.0,100000,5387,0.054,0.001,0.052,0.055


In [28]:
print("Innovations on raw predicted states (Monte Carlo averages and rejection rates):")
_sort_summary(mc_reference["measurement_regressions_raw_summary"]).round(3)

Innovations on raw predicted states (Monte Carlo averages and rejection rates):


,measurement,predictor,coef,standardized_coef,std_error,t_stat,p_value,r2,mc_se_coef,mc_se_standardized_coef,mc_se_std_error,mc_se_t_stat,mc_se_p_value,mc_se_r2,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high
2,OutGap,Pi,-3.054,-0.258,0.807,-3.773,0.004,0.069,0.002,0.0,0.000,0.003,0.000,0.0,100000,98498,0.985,0.000,0.984,0.986
1,OutGap,x,-0.414,-0.240,0.117,-3.501,0.007,0.061,0.000,0.0,0.000,0.003,0.000,0.0,100000,96689,0.967,0.001,0.966,0.968
0,OutGap,r,0.599,0.021,1.945,0.298,0.576,0.003,0.005,0.0,0.001,0.002,0.001,0.0,100000,1165,0.012,0.000,0.011,0.012
5,Infl,Pi,-0.163,-0.010,0.982,-0.148,0.497,0.005,0.003,0.0,0.000,0.003,0.001,0.0,100000,5303,0.053,0.001,0.052,0.054
4,Infl,x,-0.021,-0.008,0.142,-0.112,0.498,0.005,0.000,0.0,0.000,0.003,0.001,0.0,100000,5040,0.050,0.001,0.049,0.052
3,Infl,r,-0.144,-0.004,2.286,-0.053,0.500,0.005,0.007,0.0,0.001,0.003,0.001,0.0,100000,4984,0.050,0.001,0.049,0.051
8,Rate,Pi,0.031,0.011,0.182,0.157,0.500,0.005,0.001,0.0,0.000,0.003,0.001,0.0,100000,5054,0.051,0.001,0.049,0.052
7,Rate,x,0.007,0.017,0.026,0.237,0.494,0.005,0.000,0.0,0.000,0.003,0.001,0.0,100000,5406,0.054,0.001,0.053,0.055
6,Rate,r,-0.044,-0.005,0.424,-0.071,0.502,0.005,0.001,0.0,0.000,0.003,0.001,0.0,100000,4937,0.049,0.001,0.048,0.051


In [34]:
print("Innovation decomposition orthogonal summary:")
_sort_summary(mc_reference["innovation_decomposition_orthogonalized_summary"])

Innovation decomposition orthogonal summary:


,measurement,predictor,beta_measurement_error,beta_state_prediction_error,beta_total_innovation,beta_component_sum,beta_component_gap,abs_beta_component_gap,reconstruction_max_abs_error,mc_se_beta_measurement_error,mc_se_beta_state_prediction_error,mc_se_beta_total_innovation,mc_se_beta_component_sum,mc_se_beta_component_gap,mc_se_abs_beta_component_gap,mc_se_reconstruction_max_abs_error
0,OutGap,Pi,1.855786,-4.089466,-2.233680,-2.233680,-3.124925e-18,5.346402e-16,1.752937e-15,0.003217,0.002355,0.005096,0.005096,2.212982e-18,1.427921e-18,8.975420e-19
1,OutGap,x,-0.003202,-0.164772,-0.167974,-0.167974,1.420037e-19,5.466518e-17,1.752937e-15,0.000478,0.000363,0.000768,0.000768,2.322132e-19,1.550485e-19,8.975420e-19
2,OutGap,r,-0.185563,-0.853644,-1.039207,-1.039207,7.780128e-19,4.431508e-16,1.752937e-15,0.003961,0.002880,0.006283,0.006283,1.873781e-18,1.243871e-18,8.975420e-19
3,Infl,Pi,-0.012537,-0.309115,-0.321652,-0.321652,-1.969967e-17,4.730421e-16,1.752937e-15,0.000652,0.006336,0.006362,0.006362,1.966279e-18,1.277664e-18,8.975420e-19
4,Infl,x,0.001663,-0.002764,-0.001101,-0.001101,4.093052e-19,6.968602e-17,1.752937e-15,0.000096,0.000942,0.000945,0.000945,2.906108e-19,1.894583e-19,8.975420e-19
5,Infl,r,-0.006908,-0.364565,-0.371473,-0.371473,1.215552e-18,5.755725e-16,1.752937e-15,0.000798,0.007805,0.007834,0.007834,2.402972e-18,1.568891e-18,8.975420e-19
6,Rate,Pi,0.000055,-0.020320,-0.020265,-0.020265,-6.149271e-19,1.751523e-16,1.752937e-15,0.000140,0.001168,0.001170,0.001170,7.014266e-19,4.303679e-19,8.975420e-19
7,Rate,x,-0.000034,0.012313,0.012279,0.012279,4.964994e-20,2.582494e-17,1.752937e-15,0.000021,0.000175,0.000175,0.000175,1.037235e-19,6.394722e-20,8.975420e-19
8,Rate,r,-0.002030,-0.014014,-0.016043,-0.016043,1.058665e-18,2.208184e-16,1.752937e-15,0.000171,0.001447,0.001444,0.001444,8.883101e-19,5.490841e-19,8.975420e-19


In [35]:
print("Innovation decomposition raw summary:")
_sort_summary(mc_reference["innovation_decomposition_raw_summary"])

Innovation decomposition raw summary:


,measurement,predictor,beta_measurement_error,beta_state_prediction_error,beta_total_innovation,beta_component_sum,beta_component_gap,abs_beta_component_gap,reconstruction_max_abs_error,mc_se_beta_measurement_error,mc_se_beta_state_prediction_error,mc_se_beta_total_innovation,mc_se_beta_component_sum,mc_se_beta_component_gap,mc_se_abs_beta_component_gap,mc_se_reconstruction_max_abs_error
0,OutGap,Pi,1.918703,-4.972770,-3.054067,-3.054067,-1.489364e-18,5.305445e-16,1.752937e-15,0.001574,0.000907,0.002319,0.002319,2.194334e-18,1.414327e-18,8.975420e-19
1,OutGap,x,0.238009,-0.652501,-0.414491,-0.414491,-4.331171e-20,6.865833e-17,1.752937e-15,0.000227,0.000223,0.000368,0.000368,2.873628e-19,1.882479e-19,8.975420e-19
2,OutGap,r,-0.907156,1.506609,0.599453,0.599453,-2.421744e-18,4.349408e-16,1.752937e-15,0.004810,0.003284,0.004560,0.004560,1.803415e-18,1.166453e-18,8.975420e-19
3,Infl,Pi,-0.000958,-0.162091,-0.163050,-0.163050,-1.894124e-17,2.366945e-16,1.752937e-15,0.000325,0.003098,0.003111,0.003111,9.771092e-19,6.309370e-19,8.975420e-19
4,Infl,x,0.000085,-0.020820,-0.020734,-0.020734,-2.410452e-18,3.418793e-17,1.752937e-15,0.000047,0.000450,0.000452,0.000452,1.414958e-19,9.160189e-20,8.975420e-19
5,Infl,r,-0.005415,-0.138924,-0.144339,-0.144339,7.811522e-18,5.465474e-16,1.752937e-15,0.000758,0.007253,0.007277,0.007277,2.273667e-18,1.477499e-18,8.975420e-19
6,Rate,Pi,-0.000004,0.031436,0.031432,0.031432,3.478830e-19,8.755808e-17,1.752937e-15,0.000070,0.000568,0.000571,0.000571,3.500151e-19,2.141187e-19,8.975420e-19
7,Rate,x,0.000001,0.007061,0.007063,0.007063,6.090112e-20,1.268492e-17,1.752937e-15,0.000010,0.000083,0.000083,0.000083,5.086187e-20,3.127104e-20,8.975420e-19
8,Rate,r,-0.001318,-0.042612,-0.043930,-0.043930,-7.728337e-20,2.115180e-16,1.752937e-15,0.000162,0.001351,0.001349,0.001349,8.482718e-19,5.216915e-19,8.975420e-19


Monte Carlo summaries above aggregate `_MC_SAMPLES` independent draws. The plots and MCMC output below continue on a representative first draw.

In [10]:
parser_aug = ModelParser(_AUGMENTED_CONFIG)
config_aug, kalman_aug = parser_aug.get_all()
solver_aug = DSGESolver(config_aug, kalman_aug)
comp_aug = solver_aug.compile(
    n_state=3,
    n_exog=3,
)
priors = {
    _AUGMENTED_PARAM: make_prior(
        'normal',
        parameters={"mean": 0.0, "std": 4.0, "random_state": next_seed(_plot_seed_counter)},
        transform="identity",
    ),
}

with STDOUT_VOID():
    mc_aug = summarize_mle_augmentation_experiment(
        sol,
        solver_aug,
        comp_aug,
        sol_dgp,
        mc_reference,
        T=T,
        candidate_param=_AUGMENTED_PARAM,
        mc_samples=_MC_SAMPLES,
        alpha=_MC_ALPHA,
    )



## Diagnostics of the Augmented Model

### Marginal LR Test Conditional on $\theta_0$


In [11]:
print("Monte Carlo LR summary for the MLE-augmented model:")
rep_aug = mc_aug["representative"]
res_mle = rep_aug.res_mle
sol_mle = rep_aug.sol_mle
mle_aug_kf = rep_aug.kf_aug
std_innov_aug_mle = rep_aug.std_innov_aug
sim_aug_mle = rep_aug.sim_aug
mc_aug["lr_summary"].round(3)

Monte Carlo LR summary for the MLE-augmented model:


,estimated_coef,loglik_ref,loglik_aug,lr,p_value,mc_se_estimated_coef,mc_se_loglik_ref,mc_se_loglik_aug,mc_se_lr,mc_se_p_value,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high
0,2.0,-1646.085,-882.392,1527.386,0.0,0.0,0.324,0.055,0.587,0.0,100000,100000,1.0,0.0,1.0,1.0


In [12]:
res_mle

OptimizationResult(kind='mle', x=array([1.92000302]), theta={'beta': np.float64(0.971), 'kappa': np.float64(0.58), 'tau_inv': np.float64(1.86), 'psi_pi': np.float64(2.19), 'psi_x': np.float64(0.3), 'rho_r': np.float64(0.84), 'rho_g': np.float64(0.83), 'rho_z': np.float64(0.85), 'pi_star': np.float64(3.43), 'r_star': np.float64(3.01), 'sig_r': np.float64(0.18), 'sig_g': np.float64(0.18), 'sig_z': np.float64(0.64), 'rho_gz': np.float64(0.36), 'meas_infl': np.float64(1e-06), 'meas_rate': np.float64(1e-06), 'meas_outgap': np.float64(1e-06), 'meas_rho_ir': np.float64(0.0), 'meas_rho_gi': np.float64(0.0), 'meas_rho_gr': np.float64(0.0), 'Pi_coef': np.float64(1.9200030238935968), 'x_coef': np.float64(0.0), 'r_coef': np.float64(0.0)}, success=True, message='CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH', fun=np.float64(853.0554706023761), loglik=np.float64(-853.0554706023761), logprior=np.float64(0.0), logpost=np.float64(-853.0554706023761), nfev=14, nit=6, raw=  message: CONVERGENCE: R

## Serial Autocorrelation Tests for the Augmented Model

In [13]:
print("Monte Carlo Ljung-Box summary for the MLE-augmented model:")
mc_aug["lb_summary"].round(3)

Monte Carlo Ljung-Box summary for the MLE-augmented model:


,measurement,lb_stat,p_value,mc_se_lb_stat,mc_se_p_value,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high
0,OutGap,1.004,0.498,0.004,0.001,100000,5006,0.050,0.001,0.049,0.051
1,Infl,1.003,0.499,0.004,0.001,100000,5059,0.051,0.001,0.049,0.052
2,Rate,1.005,0.498,0.004,0.001,100000,5048,0.050,0.001,0.049,0.052


In [14]:
print("Reference moment-specification test summary:")
display(mc_reference["moment_specification_test_summary"].round(3))

print("Augmented moment-specification test summary:")
display(mc_aug["moment_specification_test_summary"].round(3))

print("Reference-minus-augmented moment distance comparison:")
display(mc_aug["moment_specification_comparison"].round(3))

Reference moment-specification test summary:


,test,distance,stat,p_value,mc_se_distance,mc_se_stat,mc_se_p_value,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high,df,sample_size,bandwidth
0,mean_zero_hac,0.120,2.860,0.535,0.000,0.009,0.001,100000,5673,0.057,0.001,0.055,0.058,3.0,200,4
1,cov_identity,8.567,1022.581,0.000,0.003,1.334,0.000,100000,100000,1.000,0.000,1.000,1.000,6.0,200,4


Augmented moment-specification test summary:


,test,distance,stat,p_value,mc_se_distance,mc_se_stat,mc_se_p_value,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high,df,sample_size,bandwidth
0,mean_zero_hac,0.113,3.304,0.475,0.0,0.009,0.001,100000,7302,0.073,0.001,0.071,0.075,3.0,200,4
1,cov_identity,0.216,6.885,0.480,0.0,0.017,0.001,100000,11627,0.116,0.001,0.114,0.118,6.0,200,4


Reference-minus-augmented moment distance comparison:


""
